[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/225/blob/main/10_pandas_sql/10_5_JOIN.ipynb)

# 10.5: `JOIN`: Combining Tables

In notebooks 10.3 and 10.4, every query that needed the `continent` column ran against the flat `gapminder` table, where continent is repeated on every row. That worked, but it is not how real databases store data. Information that repeats, like which continent a country belongs to, is normally stored once in its own table and combined in when needed. Our database already contains that split: `measurements` holds the year-by-year observations but has no continent column, and `countries` holds the country-to-continent mapping. To answer a question like "what is the mean life expectancy per continent in 2007?" from the split tables, you need to bring them together.

In pandas, combining two DataFrames on a shared key is called `merge()`. In SQL, the same operation is called a **`JOIN`**. When the key columns have no missing values, the two give the same rows. This notebook covers the two JOIN types you will use in almost every real query: `INNER JOIN` and `LEFT JOIN`.

In [ ]:
import pandas as pd
import sqlite3

url = "https://raw.githubusercontent.com/jennybc/gapminder/main/inst/extdata/gapminder.tsv"
df = pd.read_csv(url, sep="\t")

countries = df[["country", "continent"]].drop_duplicates().reset_index(drop=True)
measurements = df[["country", "year", "lifeExp", "pop", "gdpPercap"]].copy()

conn = sqlite3.connect(":memory:")
countries.to_sql("countries", conn, index=False, if_exists="replace")
measurements.to_sql("measurements", conn, index=False, if_exists="replace")
df.to_sql("gapminder", conn, index=False, if_exists="replace")

print(f"countries: {countries.shape}")
print(f"measurements: {measurements.shape}")

## Why two tables?

The Gapminder data started as one flat file. Splitting it into two tables is artificial here; in the original file, continent is on every row. But this split mimics how data looks in real databases, where redundant information is stored once and referenced by a key rather than repeated in every row.

In a sales database, for example, each order record might contain a `customer_id` but not the customer's name and address; those live in a separate `customers` table and are joined in when needed. The benefit is that if a customer's address changes, it changes in one place, not in thousands of order records. The cost is that answering questions often requires combining tables.

In our case: `measurements` has `country` but not `continent`. `countries` has the mapping. To group measurements by continent, we need to join them.

## The pandas version: `merge()`

You have seen `merge()` only in passing, in a module 09 exercise solution, so here it is properly before the SQL. `left.merge(right, on="key")` matches each row of `left` with the rows of `right` that have the same value in the `key` column. An optional `how=` argument decides what happens to rows with no match; the default, `how="inner"`, drops them.

In [ ]:
combined_pandas = measurements.merge(countries, on="country")
print(f"Shape after merge: {combined_pandas.shape}")
combined_pandas.head()

1,704 rows, matching every row from `measurements` to its corresponding row in `countries` on the shared `country` column. The result has all the original columns from both tables. This is what a JOIN does: it finds rows in two tables that share the same value in a specified column and combines them into a single wider row.

## `INNER JOIN`: the SQL equivalent of `merge(how="inner")`

The SQL syntax for a JOIN puts the two tables in `FROM` and `JOIN`, connected by an `ON` clause that names the shared key. Table aliases (`AS m`, `AS c`) shorten the table names to a single letter so column references (`m.country`, `c.continent`) stay readable.

In [ ]:
pd.read_sql("""
    SELECT m.country,
           c.continent,
           m.year,
           m.lifeExp,
           m.gdpPercap
    FROM measurements AS m
    INNER JOIN countries AS c ON m.country = c.country
    LIMIT 8
""", conn)

The result contains the same matched rows as the pandas merge: `continent` from `countries` has been brought in alongside the measurement columns. The `ON m.country = c.country` clause is the join condition; it specifies which column in each table holds the matching key.

Both tables have a `country` column, so a bare `country` in this query would be ambiguous and the database would raise an error. Prefixing the column with its table removes the ambiguity. You could write the full table name (`measurements.country`), but the short aliases (`AS m`, `AS c`) make the prefixes much easier to type and read, which is why they are standard whenever a query uses more than one table.

| pandas | SQL |
|---|---|
| `measurements.merge(countries, on="country")` | `FROM measurements AS m INNER JOIN countries AS c ON m.country = c.country` |

## What `INNER JOIN` drops

An INNER JOIN keeps only rows where the key exists in both tables. Any row in either table with no matching row in the other is silently dropped. To make this concrete, let's add a country to `measurements` that does not exist in `countries`. The next cell uses `pd.concat()`, which stacks one DataFrame under another, to add the row.

In [ ]:
# Add a fake country to measurements
fake_row = pd.DataFrame([{"country": "Narnia", "year": 2007,
                           "lifeExp": 999.0, "pop": 1000, "gdpPercap": 50000.0}])
measurements_with_fake = pd.concat([measurements, fake_row], ignore_index=True)
measurements_with_fake.to_sql("measurements", conn, index=False, if_exists="replace")

# INNER JOIN: Narnia disappears because it has no match in countries
result_inner = pd.read_sql("""
    SELECT m.country, c.continent, m.year, m.lifeExp
    FROM measurements AS m
    INNER JOIN countries AS c ON m.country = c.country
    WHERE m.country IN ('Afghanistan', 'Narnia')
""", conn)
print("INNER JOIN result:")
result_inner

Narnia's row is not in the result. The INNER JOIN found no matching row in `countries` for `country = 'Narnia'`, so the entire row was excluded from the result. Afghanistan's 12 rows are all present because Afghanistan appears in both tables.

This behavior is usually what you want, since you are asking for data that exists in both tables. But sometimes a missing match is meaningful information: a measurement with no country record, or an order with no customer. For those cases, you need `LEFT JOIN`.

## `LEFT JOIN`: keeping all rows from the left table

A `LEFT JOIN` keeps every row from the left table (the one in `FROM`), even if there is no matching row in the right table. Where there is no match, the columns from the right table are filled with `NULL`. This is the SQL equivalent of `merge(how="left")`.

In [ ]:
# LEFT JOIN: Narnia stays, but its continent is NULL
result_left = pd.read_sql("""
    SELECT m.country, c.continent, m.year, m.lifeExp
    FROM measurements AS m
    LEFT JOIN countries AS c ON m.country = c.country
    WHERE m.country IN ('Afghanistan', 'Narnia')
""", conn)
print("LEFT JOIN result:")
result_left

Now Narnia appears in the result, but its `continent` is missing because the `countries` table has no row for it. In the database the value is `NULL`, SQL's marker for a missing value, and pandas shows it as `NaN` or `None`, depending on the pandas version. The left table (`measurements`) was preserved in full; the right table (`countries`) contributed what it could.

This is how you use LEFT JOIN to find unmatched rows: filter the result for `WHERE c.continent IS NULL` and you get only the rows in `measurements` that had no match in `countries`. You have to write `IS NULL`, because `c.continent = NULL` finds nothing: in SQL a comparison with `NULL` is never true, even a comparison of `NULL` with `NULL`.

| pandas | SQL |
|---|---|
| `measurements.merge(countries, on="country", how="inner")` | `INNER JOIN` |
| `measurements.merge(countries, on="country", how="left")` | `LEFT JOIN` |

The Narnia row added above is still in the database. The cell below overwrites the `measurements` table with the original data so the remaining examples work with real Gapminder observations.

In [ ]:
# Restore the real measurements table (without Narnia)
measurements.to_sql("measurements", conn, index=False, if_exists="replace")
print("Measurements restored.")

## JOIN + WHERE + GROUP BY together

In notebooks 10.3 and 10.4, the queries that needed `continent` used the flat `gapminder` table. With JOIN, the same kind of question can be answered from the split tables. Here is a full example that chains four clauses: JOIN to combine tables, WHERE to filter rows, GROUP BY to aggregate, HAVING to filter groups.

In [ ]:
pd.read_sql("""
    SELECT c.continent,
           COUNT(DISTINCT m.country) AS n_countries,
           ROUND(AVG(m.lifeExp), 1) AS avg_life,
           ROUND(AVG(m.gdpPercap), 0) AS avg_gdp
    FROM measurements AS m
    INNER JOIN countries AS c ON m.country = c.country
    WHERE m.year = 2007
    GROUP BY c.continent
    HAVING COUNT(DISTINCT m.country) >= 10
    ORDER BY avg_life DESC
""", conn)

Oceania (2 countries) is excluded by the `HAVING COUNT(DISTINCT m.country) >= 10` condition. The remaining four continents each have at least 25 countries. The query reads like a plain-English description of the analysis: combine the tables, keep only 2007 data, compute per-continent summaries, drop continents with fewer than 10 countries, sort by life expectancy.

## A join can multiply rows

That query used `COUNT(DISTINCT m.country)` rather than `COUNT(*)`, and the reason is worth seeing. Each row of `countries` matches many rows of `measurements`, so a join repeats each country once for every match.

In [ ]:
pd.read_sql("""
    SELECT c.continent,
           COUNT(*) AS rows_after_join,
           COUNT(DISTINCT c.country) AS n_countries
    FROM countries AS c
    INNER JOIN measurements AS m ON c.country = m.country
    GROUP BY c.continent
    ORDER BY n_countries DESC
""", conn)

Africa has 52 countries but 624 rows after the join, because each country now appears once for each of its 12 years. This changes the answer whenever you aggregate a column that describes the country and not the year. If `countries` had a column for land area, `SUM(c.area)` after this join would add each country's area 12 times, with no error or warning. So before you add up or count anything after a join, ask what one row of the joined result represents. Comparing the row count before and after a join is a quick check.

## Missing values: where SQL and pandas differ

Gapminder has no missing values, so SQL and pandas have agreed throughout this module. On data with missing values they can give different answers:

- `WHERE col != 'a'` drops rows where `col` is `NULL`, while `df.query("col != 'a'")` keeps rows where it is `NaN`. `NOT IN` and `~isin()` differ in the same way.
- `GROUP BY` keeps a group for `NULL` keys, while `groupby()` drops `NaN` keys by default.
- A join does not match `NULL` to `NULL`, while `merge()` does match `NaN` to `NaN`.
- `SUM` of a group with only `NULL` values is `NULL`, while pandas `.sum()` gives 0.

When a column has missing values, check how many rows each step keeps rather than assuming the two tools agree.

## A note on JOIN types

SQL has four main JOIN types: `INNER JOIN`, `LEFT JOIN`, `RIGHT JOIN`, and `FULL OUTER JOIN`. In practice, `RIGHT JOIN` and `FULL OUTER JOIN` are rarely used in analytical queries: a `RIGHT JOIN` is just a `LEFT JOIN` with the tables swapped, and the situations that genuinely need a `FULL OUTER JOIN` are uncommon in analysis work. SQLite did not support them at all until 2022. Learning `INNER` and `LEFT` covers the large majority of the joins you will write in analysis work.

The complete clause order for a SQL query is now:

```
SELECT   ...
FROM     ...
JOIN     ... ON ...
WHERE    ...
GROUP BY ...
HAVING   ...
ORDER BY ...
LIMIT    ...
```

Not every query uses every clause, but when multiple clauses appear, they must come in this order.

## Updated translation table

| Goal | pandas | SQL |
|---|---|---|
| Select columns | `df[["a", "b"]]` | `SELECT a, b FROM table` |
| All columns | `df` | `SELECT * FROM table` |
| Rename column | `df.rename(columns={"old": "new"})` | `SELECT old AS new FROM table` |
| First N rows | `df.head(N)` | `LIMIT N` |
| Unique values | `df["col"].unique()` | `SELECT DISTINCT col FROM table` |
| Sort ascending | `df.sort_values("col")` | `ORDER BY col ASC` |
| Sort descending | `df.sort_values("col", ascending=False)` | `ORDER BY col DESC` |
| Filter rows | `df.query("col > x")` | `WHERE col > x` |
| Match equality | `df.query("col == 'val'")` | `WHERE col = 'val'` |
| Match list | `df["col"].isin([...])` | `WHERE col IN (...)` |
| Exclude list | `~df["col"].isin([...])` | `WHERE col NOT IN (...)` |
| Range filter | `df.query("lo <= col <= hi")` | `WHERE col BETWEEN lo AND hi` |
| String starts with | `df["col"].str.startswith("A")` | `WHERE col LIKE 'A%'` |
| Count rows per group | `groupby(col).size()` | `COUNT(*) ... GROUP BY col` |
| Count non-null | `groupby(col)["c"].count()` | `COUNT(c) ... GROUP BY col` |
| Count distinct | `.nunique()` | `COUNT(DISTINCT col)` |
| Mean per group | `groupby(col)["v"].mean()` | `AVG(v) ... GROUP BY col` |
| Sum per group | `groupby(col)["v"].sum()` | `SUM(v) ... GROUP BY col` |
| Min / Max per group | `.min()` / `.max()` | `MIN(v)` / `MAX(v)` |
| Multiple aggregations | `.agg(["mean", "min"])` | `AVG(v), MIN(v)` in SELECT |
| Filter groups | `groupby(col).agg(...)`, then filter the result | `HAVING condition` |
| Combine tables (inner) | `df1.merge(df2, on="key")` | `INNER JOIN t2 ON t1.key = t2.key` |
| Combine tables (left) | `df1.merge(df2, on="key", how="left")` | `LEFT JOIN t2 ON t1.key = t2.key` |
| Find missing values | `df[df["col"].isna()]` | `WHERE col IS NULL` |

## What's next

You now have the main SQL clauses for analytical queries: SELECT, WHERE, GROUP BY, HAVING, and JOIN. Notebook 10.6 introduces one more concept, subqueries, where one SELECT is nested inside another, and then steps back to compare SQL and pandas as tools: what each does best, and how to choose between them for a given problem.